In [ ]:
#Code checks the Principal Components of various sizes of GAF images
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

#Directories for data and GAF images
data_dir = r'\ProcessedDataNumpy_norm_exoatmoshperic'
gaf_dirs = [
    r'18x18_norm_exoatmoshperic',
    r"32x32_norm_exoatmospheric",
    r'64x64_norm_exoatmoshperic',
    r'128x128_norm_exoatmoshperic'
]

#Loading data
all_norm_data = []
all_labels = []

for file_name in os.listdir(data_dir):
    if file_name.endswith('.npy'):
        file_path = os.path.join(data_dir, file_name)
        data = np.load(file_path, allow_pickle=True).item()
        
        norm_data = data['Norm']
        satellite_name = file_name.split('_')[0]  #Extract class name from file name
        
        for measurement in norm_data:
            all_norm_data.append(measurement)
            all_labels.append(satellite_name)

all_norm_data = np.array(all_norm_data)
all_labels = np.array(all_labels)

if np.any(np.isnan(all_norm_data)) or np.any(np.isinf(all_norm_data)):
    raise ValueError("Data contains NaN or infinite values.")

label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)

#Choose one example to plot the PCA results, indexed from 0-19
encoded_value_to_plot = 5  
chosen_label = label_encoder.inverse_transform([encoded_value_to_plot])[0]#Get the original label from the encoded value

def apply_pca_and_plot(data, label, title_suffix):
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    pca = PCA()
    data_pca = pca.fit_transform(data_scaled)
    
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_explained_variance = np.cumsum(explained_variance_ratio)
    
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.bar(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, alpha=0.7, align='center', label='Individual explained variance')
    plt.step(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, where='mid', label='Cumulative explained variance')
    plt.xlabel('Principal Component Index')
    plt.ylabel('Explained Variance Ratio')
    plt.title(f'Explained Variance Ratio by Principal Component for {label} {title_suffix}')
    plt.legend(loc='best')
    
    plt.subplot(1, 2, 2)
    plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o', linestyle='--', color='b')
    plt.xlabel('Number of Principal Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.title(f'Cumulative Explained Variance for {label} {title_suffix}')
    plt.grid()
    
    plt.tight_layout()
    plt.show()

#Apply PCA to the GAF images for the chosen class label
for gaf_dir in gaf_dirs:
    gaf_size = gaf_dir.split('_')[1]
    gaf_class_data = []
    for file_name in os.listdir(gaf_dir):
        if file_name.endswith('.npy') and file_name.startswith(chosen_label):
            file_path = os.path.join(gaf_dir, file_name)
            gaf_image = np.load(file_path)
            gaf_class_data.append(gaf_image.flatten())
    gaf_class_data = np.array(gaf_class_data)
    apply_pca_and_plot(gaf_class_data, chosen_label, f'({gaf_size} GAF Images)')

In [ ]:
#This is the code for the hyperparameter search 
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from itertools import product
from sklearn.model_selection import KFold
from optuna.visualization import plot_param_importances
from sklearn.manifold import TSNE
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Initialize data
data_dir = r"/home/jtk1704/SPRING2025/NonSolarAnalogData/GramianImage_18x18_norm_exoatmoshperic/"
saved_images = [f for f in os.listdir(data_dir) if f.endswith('.npy')]
X_gaf, all_labels = [], []
for file_name in saved_images:
    gaf_image = np.load(os.path.join(data_dir, file_name))
    X_gaf.append(gaf_image)
    label = file_name.split('_')[0].lower()
    all_labels.append(label)
X_gaf = np.array(X_gaf).reshape(-1, 1, 18, 18)

label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)

X_train, X_test, y_train, y_test = train_test_split(X_gaf, all_labels_encoded, test_size=0.2, random_state=42)

#DataLoader prep
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

def one_hot_encode(labels, num_classes):
    return torch.eye(num_classes, device=labels.device)[labels]#.to(labels.device)
 
#CVAE Model
class CVAE(nn.Module):
    def __init__(self, input_channels=1, hidden_dim=512, latent_dim=50, num_classes=20, noise_level=0.3, conv_filters0=64,conv_filters1 = 128, conv_filters2 = 256, conv_filters3 = 512, activation_function=nn.PReLU(), dropout_rate=0.3):
        super(CVAE, self).__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        self.noise_level = noise_level 
        self.conv_filters0 = conv_filters0
        self.conv_filters1 = conv_filters1
        self.conv_filters2 = conv_filters2
        self.conv_filters3 = conv_filters3
        self.actfunct = activation_function
        self.dropout_rate = dropout_rate

        #Encoder
        self.conv1 = nn.Conv2d(input_channels + num_classes, conv_filters1, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(conv_filters1)
        self.conv2 = nn.Conv2d(conv_filters1, conv_filters2 * 2, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(conv_filters2 * 2)
        self.conv3 = nn.Conv2d(conv_filters2 * 2, conv_filters3 * 4, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(conv_filters3 * 4)

        self.fc_mu = nn.Linear(conv_filters3 * 4 * 3 * 3, latent_dim)
        self.fc_logvar = nn.Linear(conv_filters3 * 4 * 3 * 3, latent_dim)

        #Decoder
        self.fc3 = nn.Linear(latent_dim + num_classes, conv_filters3 * 4 * 3 * 3)
        self.deconv1 = nn.ConvTranspose2d(conv_filters3 * 4, conv_filters2 * 2, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn4 = nn.BatchNorm2d(conv_filters2 * 2)
        self.deconv2 = nn.ConvTranspose2d(conv_filters2 * 2, conv_filters1, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn5 = nn.BatchNorm2d(conv_filters1)
        self.deconv3 = nn.ConvTranspose2d(conv_filters1, conv_filters0, kernel_size=2, stride=2, padding=3, output_padding=0)
        self.bn6 = nn.BatchNorm2d(conv_filters0)
        self.deconv4 = nn.ConvTranspose2d(conv_filters0, input_channels, kernel_size=3, stride=1, padding=1)
        self.dropout = nn.Dropout(dropout_rate)

        #The linear classifer was used here due to the ltent space being well structured
        self.classifier = nn.Linear(latent_dim, num_classes)
    
    def encode(self, x, c):#Had some issues with this and so I did the following:
        #c = c.view(-1, self.num_classes, 1, 1).expand(-1, -1, x.size(2), x.size(3))
        #x = torch.cat([x, c], dim=1)
        # One-hot encode the labels
        c_one_hot = one_hot_encode(c, self.num_classes)  # Shape: [batch_size, num_classes]
    
    #Reshape and expand to match the input dimensions
        c_one_hot = c_one_hot.view(-1, self.num_classes, 1, 1).expand(-1, -1, x.size(2), x.size(3))
    
    #Concatenate the input and the one-hot encoded labels
        x = torch.cat([x, c_one_hot], dim=1)
        h1 = self.actfunct(self.bn1(self.conv1(x)))
        h2 = self.actfunct(self.bn2(self.conv2(h1)))
        h3 = self.actfunct(self.bn3(self.conv3(h2)))
        h3 = h3.view(h3.size(0), -1)
        h3 = self.dropout(h3)
        mu = self.fc_mu(h3)
        logvar = self.fc_logvar(h3)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)* self.noise_level #here?
        return mu + eps * std

    def decode(self, z, c):
        #z = torch.cat([z, c], dim=1)
        c_one_hot = one_hot_encode(c, self.num_classes)  #Shape: [batch_size, num_classes]
    
        #Concatenate the latent vector z and the one-hot encoded labels
        z = torch.cat([z, c_one_hot], dim=1)  #Shape: [batch_size, latent_dim + num_classes]
    
        h3 = self.actfunct(self.fc3(z))
        h3 = h3.view(h3.size(0), self.conv_filters3 * 4, 3, 3)
        h4 = self.actfunct(self.bn4(self.deconv1(h3)))
        h5 = self.actfunct(self.bn5(self.deconv2(h4)))
        h6 = self.actfunct(self.bn6(self.deconv3(h5)))
        h7 = torch.tanh(self.deconv4(h6)) #tanh due to the -1 to 1 range
        return h7

    def forward(self, x, c):
        c_one_hot = one_hot_encode(c, self.num_classes)  #I wonder if soft labels would work better for my data, maybe in the CNN later on
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        decoded = self.decode(z, c)
        logits = self.classifier(z)  #Predict class logits
        return decoded, mu, logvar, logits
        

#Penalty term based on MSE difference between decoded images. Introduces variance during training by penalizing the difference between consecutive decoded images.
#Model focuses on instances that display borderline or overlapping characteristics, enhancing its robustness.
def enhanced_loss_function(recon_x, x, mu, logvar, logits, labels, beta=1.0, gamma=0.1):
    RE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    penalty_term = 0
    penalty_term = nn.functional.mse_loss(recon_x[:-1], recon_x[1:])#for i in range(len(recon_x) - 1): #using what was advised instead of what i had initially. the fix should make the code more efficient 
        #penalty_term += nn.functional.mse_loss(recon_x[i], recon_x[i + 1])
    #Class-balanced cross-entropy
    label_indices = labels
    #label_indices = torch.argmax(labels, dim=1) #advised to do the following instead: labels_indices = labels
    total_classes = logits.shape[1]
    #Class-balanced loss  Ensures that the model pays attention to underrepresented classes during training. This is currently a placeholder as it adds 1to the class weights for each class.
    #By minimizing this loss, the model ensures that it learns to generate balanced representations across all classes, addressing class imbalance when properly configured.
    class_weights = torch.ones(total_classes, device=label_indices.device)
    class_weights[label_indices] += 1.0
    class_balanced_loss = nn.functional.cross_entropy(logits, label_indices, weight=class_weights)
    total_loss = RE + beta * KLD + gamma * penalty_term + class_balanced_loss
    return total_loss, RE, KLD, penalty_term, class_balanced_loss

#Objective function
def objective(trial):
    # Suggest hyperparameters
    hidden_dim = trial.suggest_int('hidden_dim', 512, 2048, step=512)
    latent_dim = trial.suggest_int('latent_dim', 100, 300, step = 2)
    learning_rate = trial.suggest_loguniform('learning_rate', 0.00001, 0.001)
    conv_filters0 = trial.suggest_int('conv_filters0', 64, 256, step=4)
    conv_filters1 = trial.suggest_int('conv_filters1', 64, 256, step=4)
    conv_filters2 = trial.suggest_int('conv_filters2', 64, 512, step=4)
    conv_filters3 = trial.suggest_int('conv_filters3', 64, 512, step=4)
    #conv_filters2 = trial.suggest_uniform('conv_filters2', 64, 1024)#showing what it used to look like conv_filters2 = trial.suggest_categorical('conv_filters2', [64, 128, 256, 512, 1024])
    #conv_filters3 = trial.suggest_uniform('conv_filters3', 64, 1024)
    dropout_rate = trial.suggest_uniform('dropout_rate', 0.01, 0.5)
    beta = trial.suggest_uniform('beta', 0.01, 2.0)
    noise_level = trial.suggest_uniform('noise_level', 0.01, 1.0)
    gamma = trial.suggest_uniform('gamma', 0.01, 2.0)

    #activation_function = trial.suggest_categorical('activation_function', [nn.ReLU(), nn.PReLU()])

    #Cross-validation
    kfold = KFold(n_splits=5, shuffle=True, random_state=42) 
    fold_losses = [] 
    best_epoch_weights = None
    last_epoch_weights = None

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
        #Split the data into training and validation sets for this fold
        X_train_fold = torch.tensor(X_train[train_idx], dtype=torch.float32).to(device)
        y_train_fold = torch.tensor(y_train[train_idx], dtype=torch.long).to(device)
        X_val_fold = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
        y_val_fold = torch.tensor(y_train[val_idx], dtype=torch.long).to(device)

        train_dataset = TensorDataset(X_train_fold, y_train_fold)
        val_dataset = TensorDataset(X_val_fold, y_val_fold)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

        cvae = CVAE(
            hidden_dim=hidden_dim, 
            latent_dim=latent_dim, 
            conv_filters0=conv_filters0, 
            conv_filters1=conv_filters1,
            conv_filters2=conv_filters2,
            conv_filters3=conv_filters3,
            activation_function=nn.PReLU(),
            dropout_rate=dropout_rate,
            noise_level=noise_level
        ).to(device)
        optimizer = optim.Adam(cvae.parameters(), lr=learning_rate)
        num_epochs = 100
        train_loss = 0
        best_epoch = 0

        for epoch in range(num_epochs):
            cvae.train()
            epoch_loss = 0
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device) 
                optimizer.zero_grad()
                #print(f"conv_filters3: {conv_filters3}") 
                #Forward pass
                recon_batch, mu, logvar, logits = cvae(x_batch, y_batch)

                #Sample class-specific data for DeepSMOTE
                class_data = x_batch[(y_batch == y_batch[0]).to(x_batch.device)]
                class_labels = y_batch[(y_batch == y_batch[0]).to(y_batch.device)]
                #class_data = x_batch[y_batch.argmax(dim=1) == y_batch.argmax(dim=1)[0]]#advised to do the following instead of the class data and label lines because they assume y_batch is one_hot encoded when it is actually integer-encoded: class_data = x_batch[y_batch == y_batch[0]]
#class_labels = y_batch[y_batch == y_batch[0]]
                #class_labels = y_batch[y_batch.argmax(dim=1) == y_batch.argmax(dim=1)[0]]
                mu_cls, logvar_cls = cvae.encode(class_data, class_labels)
                encoded_cls_data = cvae.reparameterize(mu_cls, logvar_cls)

                #Permute & decode. Permuted samples are used to introduce variance during training.
                permuted_cls = encoded_cls_data[torch.randperm(encoded_cls_data.size(0), device=encoded_cls_data.device)]#permuted_cls = encoded_cls_data[torch.randperm(encoded_cls_data.size(0))#]
                permuted_decoded_cls = cvae.decode(permuted_cls, class_labels)

                #Extra penalty for permuted samples
                penalty_loss = nn.functional.mse_loss(permuted_decoded_cls, class_data)

                #Combined loss
                loss, RE, KLD, penalty_term, class_balanced_loss = enhanced_loss_function(
                    recon_batch, x_batch, mu, logvar,logits, y_batch, beta=beta, gamma=gamma
                )
                loss += penalty_loss  #Add the penalty loss
                loss.backward()
                epoch_loss += loss.item()
                optimizer.step()

            avg_epoch_loss = epoch_loss / len(train_loader)
            trial.report(avg_epoch_loss, epoch)  #Report intermediate loss to Optuna for median pruner

            #Check if the trial should be pruned
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            #Update the best epoch for this trial
            if avg_epoch_loss < train_loss or epoch == 0:
                train_loss = avg_epoch_loss
                best_epoch = epoch + 1  
                best_epoch_weights = cvae.state_dict()

        #Validation loss for this fold
        cvae.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)  
                recon_batch, mu, logvar, logits = cvae(x_batch, y_batch)
                loss, RE, KLD, penalty_term, class_balanced_loss = enhanced_loss_function(
                    recon_batch, x_batch, mu, logvar, logits, y_batch, beta=beta, gamma=gamma
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)
        fold_losses.append(val_loss) 

    #Compute the average loss across all folds
    avg_fold_loss = np.mean(fold_losses)
    
    #Save the best epoch weights and last epoch weights for the best trial
    if trial.number == study.best_trial.number:
        best_epoch_weights = r"/best_epoch_optuna18x18exoatmospheric_deepsmote_model.pth"
        last_epoch_weights = r"/last_epoch_optuna18x18exoatmospheric_deepsmote_model.pth"
    
        torch.save(model.state_dict(), best_epoch_weights) 
        torch.save(cvae.state_dict(), last_epoch_weights)  
    print("\nTrial Results:")
    for t in study.trials:
        print(f"Trial {t.number}:")
        print(f"  Hyperparameters: {t.params}")
        print(f"  Best Epoch: {t.user_attrs.get('best_epoch', 'N/A')}")
        print(f"  Total Epochs: {t.user_attrs.get('num_epochs', 'N/A')}")
        print(f"  Loss: {t.value}")
    if trial.number == study.best_trial.number:
        torch.save(best_epoch_weights, "best_epoch_weights.pth")
        torch.save(cvae.state_dict(), "last_epoch_weights.pth")
    #Save the number of epochs used for this trial
    trial.set_user_attr("best_epoch", best_epoch)

    return avg_fold_loss

#Hyperparam with bayesian method study setup
study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(),
    pruner=MedianPruner()  
)
study.optimize(objective, n_trials=5)
fig1 = plot_param_importances(study) #Show parameter importances for future reference
file_path = r"param_importances.html"
fig1.write_html(file_path)
print(f"Parameter importance plot saved as '{file_path}'. Open this file in a browser to view the plot.")
best_trial = study.best_trial
best_epoch = best_trial.user_attrs["best_epoch"]
print(f"Best hyperparameters: {best_trial.params}")
print(f"Best loss: {best_trial.value}")
print(f"Best epoch: {best_epoch}")
#Initialize CVAE with the best parameters
best_params = study.best_trial.params
cvae = CVAE(
    hidden_dim=best_params['hidden_dim'],
    latent_dim=best_params['latent_dim'],
    conv_filters0=best_params['conv_filters0'],
    conv_filters1=best_params['conv_filters1'],
    conv_filters2=best_params['conv_filters2'],
    conv_filters3=best_params['conv_filters3'],
    activation_function=nn.PReLU,
    dropout_rate=best_params['dropout_rate'],
    noise_level=best_params['noise_level']
).to(device)

#Load the best weights (if saved)
best_epoch_weights_path = r"best_epoch_optuna18x18exoatmospheric_deepsmote_model.pth"
if os.path.exists(best_epoch_weights_path):
    cvae.load_state_dict(torch.load(best_epoch_weights_path))
    print("Loaded best epoch weights into CVAE.")
else:
    print("Best epoch weights file not found. Using untrained CVAE.")
#Generate synthetic samples
cvae.eval()
with torch.no_grad():
    #Random latent + random one-hot
    z = torch.randn(1000, latent_dim)
    labels = torch.eye(num_classes)[torch.randint(0, num_classes, (1000,))]
    synthetic_samples = cvae.decode(z, labels).cpu().numpy()
    synthetic_labels = np.argmax(labels.numpy(), axis=1)

#Apply SMOTE to latent space, decode after counting
unique_orig, counts_orig = np.unique(y_train, return_counts=True)
orig_counts_dict = dict(zip(unique_orig, counts_orig))

#Sampling_strategy dict for SMOTE
#sampling_strategy = {cls: 30 for cls, cnt in orig_counts_dict.items() if cnt < 30} implemented this a few lines down
from collections import Counter
original_counts = Counter(y_train)
print("Original class counts:", original_counts)
smote = SMOTE(sampling_strategy='auto', random_state=42)
encoded_train_data, _ = cvae.encode(
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(y_train_onehot, dtype=torch.float32)
)
encoded_train_data = encoded_train_data.detach().numpy()
smote_latent, smote_labels = smote.fit_resample(encoded_train_data, y_train)
decoded_smote = cvae.decode(
    torch.tensor(smote_latent, dtype=torch.float32),
    torch.tensor(np.eye(num_classes)[smote_labels], dtype=torch.float32)
).detach().numpy()

target_count = 30  #Target count for each class after SMOTE
additional_samples_needed = {cls: target_count - original_counts[cls] for cls in original_counts if original_counts[cls] < target_count}
print("Additional samples needed:", additional_samples_needed)

#Select the required number of samples from the SMOTE-generated dataset for each class
selected_smote_samples = []
selected_smote_labels = []
for cls, count in additional_samples_needed.items():
    smote_indices = np.where(smote_labels == cls)[0]
    selected_indices = np.random.choice(smote_indices, count, replace=False)
    selected_smote_samples.append(decoded_smote[selected_indices])
    selected_smote_labels.append(smote_labels[selected_indices])

#Combine the original dataset with the selected SMOTE-generated samples
X_train_combined = np.concatenate([X_train] + selected_smote_samples, axis=0)
y_train_combined = np.concatenate([y_train] + selected_smote_labels, axis=0)

unique_orig, counts_orig = np.unique(y_train, return_counts=True)
print("Original training set class counts:")
for c, num in zip(unique_orig, counts_orig):
    print(f"Class {c}, Count: {num}")

unique_comb, counts_comb = np.unique(y_train_combined, return_counts=True)
print("Combined training set class counts:")
for c, num in zip(unique_comb, counts_comb):
    print(f"Class {c}, Count: {num}")

#Flatten data before fitting RF
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

original_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
original_classifier.fit(X_train_flat, y_train)
y_pred_original = original_classifier.predict(X_test_flat)
print("Original dataset report:")
print(classification_report(y_test, y_pred_original))

X_train_combined_flat = X_train_combined.reshape(X_train_combined.shape[0], -1)
combined_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
combined_classifier.fit(X_train_combined_flat, y_train_combined)

y_pred_combined = combined_classifier.predict(X_test_flat)
print("Combined dataset report:")
print(classification_report(y_test, y_pred_combined))

#Display original and SMOTE samples side by side
selected_class = 0
original_sample_index = np.where(y_train == selected_class)[0][0]
original_sample = X_train[original_sample_index]
smote_sample_index = np.where(smote_labels == selected_class)[0][0]
smote_sample = decoded_smote[smote_sample_index]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_sample[0], cmap='gray')
axes[0].set_title('Original Sample')
axes[0].axis('off')
axes[1].imshow(smote_sample[0], cmap='gray')
axes[1].set_title('SMOTE Sample')
axes[1].axis('off')
plt.show()

#Function to plot class distribution using t-SNE
def plot_class_distribution_tsne(X, y, title):
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X)
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab20', s=50)
    plt.colorbar(scatter, ticks=range(num_classes), label='Class')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.title(title)
    plt.show()

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_train_combined_flat = X_train_combined.reshape(X_train_combined.shape[0], -1)

#Plot class distribution for the original dataset using t-SNE
plot_class_distribution_tsne(X_train_flat, y_train, 'Original Class Distribution')

#Plot class distribution for the combined dataset using t-SNE
plot_class_distribution_tsne(X_train_combined_flat, y_train_combined, 'Combined Class Distribution')

In [ ]:
#This is now the correctly working code using the parameters found from the hyperparam search for the 18x18 images (this works better than when adding contrastive loss to model)
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

data_dir = r"/18x18_norm_exoatmoshperic/"
saved_images = [f for f in os.listdir(data_dir) if f.endswith('.npy')]
X_gaf, all_labels = [], []
for file_name in saved_images:
    gaf_image = np.load(os.path.join(data_dir, file_name))
    X_gaf.append(gaf_image)
    label = file_name.split('_')[0].lower()
    all_labels.append(label)
X_gaf = np.array(X_gaf).reshape(-1, 1, 18, 18)
label_encoder = LabelEncoder()
all_labels_encoded = label_encoder.fit_transform(all_labels)
X_train, X_test, y_train, y_test = train_test_split(X_gaf, all_labels_encoded, test_size=0.2, random_state=42)

latent_dim = 170
noise_level = 0.78793139
num_classes = 20

class CVAE(nn.Module):
    def __init__(self, input_channels=1, hidden_dim=512, latent_dim=170, num_classes=20, activation_function=nn.PReLU()):
        super(CVAE, self).__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        self.actfunct = activation_function

        #Encode
        self.conv1 = nn.Conv2d(input_channels + num_classes, 212, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(212)
        self.conv2 = nn.Conv2d(212, 312, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(312)
        self.conv3 = nn.Conv2d(312, 224, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(224)

        self.fc_mu = nn.Linear(224 * 3 * 3, latent_dim)
        self.fc_logvar = nn.Linear(224 * 3 * 3, latent_dim)

        #Decode
        self.fc3 = nn.Linear(latent_dim + num_classes, 224 * 3 * 3)
        self.deconv1 = nn.ConvTranspose2d(224, 312, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn4 = nn.BatchNorm2d(312)
        self.deconv2 = nn.ConvTranspose2d(312, 212, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn5 = nn.BatchNorm2d(212)
        self.deconv3 = nn.ConvTranspose2d(212, 124, kernel_size=2, stride=2, padding=3, output_padding=0)
        self.bn6 = nn.BatchNorm2d(124)
        self.deconv4 = nn.ConvTranspose2d(124, input_channels, kernel_size=3, stride=1, padding=1)

        self.dropout = nn.Dropout(0.116629975)#(0.5)
        self.classifier = nn.Linear(latent_dim, num_classes)

    def encode(self, x, c):
        c = c.view(-1, self.num_classes, 1, 1).expand(-1, -1, x.size(2), x.size(3))
        x = torch.cat([x, c], dim=1)
        # Pass through conv layers
        h1 = self.actfunct(self.bn1(self.conv1(x)))
        h2 = self.actfunct(self.bn2(self.conv2(h1)))
        h3 = self.actfunct(self.bn3(self.conv3(h2)))
        h3 = h3.view(h3.size(0), -1)
        h3 = self.dropout(h3)
        mu = self.fc_mu(h3)
        logvar = self.fc_logvar(h3)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std) * noise_level
        return mu + eps * std

    def decode(self, z, c):
        z = torch.cat([z, c], dim=1)
        h3 = self.actfunct(self.fc3(z))
        h3 = h3.view(h3.size(0), 224, 3, 3)#512
        h4 = self.actfunct(self.bn4(self.deconv1(h3)))
        h5 = self.actfunct(self.bn5(self.deconv2(h4)))
        h6 = self.actfunct(self.bn6(self.deconv3(h5)))
        h7 = torch.tanh(self.deconv4(h6))
        return h7

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        logits = self.classifier(z)
        decoded = self.decode(z, c)
        return decoded, mu, logvar, logits

def enhanced_loss_function(recon_x, x, mu, logvar, logits, labels, beta=0.724987, gamma=0.42015935):#beta=1.0, gamma=0.1
    RE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    penalty_term = 0
    for i in range(len(recon_x) - 1):
        penalty_term += nn.functional.mse_loss(recon_x[i], recon_x[i + 1])
    label_indices = torch.argmax(labels, dim=1)
    total_classes = logits.shape[1]
    class_weights = torch.ones(total_classes, device=label_indices.device)
    class_weights[label_indices] += 1.0
    class_balanced_loss = nn.functional.cross_entropy(logits, label_indices, weight=class_weights)
    total_loss = RE + beta * KLD + gamma * penalty_term + class_balanced_loss
    return total_loss, RE, KLD, penalty_term, class_balanced_loss

#Various functions to check the performance of the model
def compute_acsa(conf_matrix):
    """
    Compute Average Class Specific Accuracy (ACSA).
    Args:
        conf_matrix: Confusion matrix (C x C).
    Returns:
        ACSA value.
    """
    per_class_accuracy = conf_matrix.diagonal() / conf_matrix.sum(axis=1)
    return np.mean(per_class_accuracy)

def compute_gm(conf_matrix):
    """
    Compute macro-averaged Geometric Mean (GM).
    Args:
        conf_matrix: Confusion matrix (C x C).
    Returns:
        GM value.
    """
    per_class_accuracy = conf_matrix.diagonal() / conf_matrix.sum(axis=1)
    return np.prod(per_class_accuracy) ** (1 / len(per_class_accuracy))

def compute_fm(conf_matrix):
    """
    Compute macro-averaged F1 measure (FM).
    Args:
        conf_matrix: Confusion matrix (C x C).
    Returns:
        FM value.
    """
    tp = conf_matrix.diagonal()
    fp = conf_matrix.sum(axis=0) - tp
    fn = conf_matrix.sum(axis=1) - tp
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1_per_class = 2 * (precision * recall) / (precision + recall)
    return np.nanmean(f1_per_class)  #Handle NaN for classes with no samples

num_folds = 5
skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

best_loss = float('inf')
best_model_path = r"/kfold_best18x18exoatmospheric_deepsmote_model.pth"
last_model_path = r"/kfold_last18x18exoatmospheric_deepsmote_model.pth"

fold = 1
for train_index, val_index in skf.split(X_train, y_train):
    print(f"Training Fold {fold}/{num_folds}")

    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    y_train_onehot = np.eye(num_classes)[y_train_fold]
    y_val_onehot = np.eye(num_classes)[y_val_fold]
    
    #DataLoaders for training and validation
    train_dataset = TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32),
                                  torch.tensor(y_train_onehot, dtype=torch.float32))
    train_loader = DataLoader(train_dataset, batch_size=15, shuffle=True)#Small batch size to assit with the DS method
    
    val_dataset = TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32),
                                torch.tensor(y_val_onehot, dtype=torch.float32))
    val_loader = DataLoader(val_dataset, batch_size=15, shuffle=False)
    
    #Initialize the CVAE and optimizer for this fold
    cvae = CVAE(input_channels=1, hidden_dim=512, latent_dim=latent_dim, num_classes=num_classes)
    optimizer = optim.Adam(cvae.parameters(), lr=0.000250157)
    
    #Train the CVAE for this fold
    num_epochs = 100
    for epoch in range(num_epochs):
        cvae.train()
        train_loss = 0

        #Step 1: Train on the entire imbalanced dataset in batches
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            #Encode → Decode → Reconstruction loss
            recon_batch, mu, logvar, logits = cvae(x_batch, y_batch)
            loss, RE, KLD, penalty_term, class_balanced_loss = enhanced_loss_function(recon_batch, x_batch, mu, logvar, logits, y_batch)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()
        
        #Step 2: Class-specific sampling and penalty loss
        y_train_onehot_tensor = torch.tensor(y_train_onehot, dtype=torch.float32)
        for class_label in range(num_classes):
            #Sample a batch of images from the same class
            class_indices = (torch.argmax(y_train_onehot_tensor, axis=1) == class_label).nonzero(as_tuple=True)[0]
            if len(class_indices) < train_loader.batch_size:
                continue  #Skip if not enough samples for the batch size
            class_data = torch.tensor(X_train_fold[class_indices[:train_loader.batch_size]], dtype=torch.float32)
            class_labels = torch.tensor(y_train_onehot_tensor[class_indices[:train_loader.batch_size]], dtype=torch.float32)

            optimizer.zero_grad()
            #Encode the class-specific data
            mu_cls, logvar_cls = cvae.encode(class_data, class_labels)
            encoded_cls_data = cvae.reparameterize(mu_cls, logvar_cls)
            #Permute the encoded data and decode
            permuted_cls = encoded_cls_data[torch.randperm(encoded_cls_data.size(0))]
            permuted_decoded_cls = cvae.decode(permuted_cls, class_labels)
            #Compute penalty loss
            penalty_loss = nn.functional.mse_loss(permuted_decoded_cls, class_data)
            penalty_loss.backward()
            train_loss += penalty_loss.item()
            optimizer.step()

        #Log training loss
        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Train Loss: {avg_train_loss}")

        cvae.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                recon_batch, mu, logvar, logits = cvae(x_batch, y_batch)
                loss, _, _, _, _ = enhanced_loss_function(recon_batch, x_batch, mu, logvar, logits, y_batch)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1}, Validation Loss: {avg_val_loss}")

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(cvae.state_dict(), best_model_path)
    
    fold += 1

torch.save(cvae.state_dict(), last_model_path)
print(f"Best model saved to {best_model_path}")
print(f"Last model saved to {last_model_path}")

#Generate synthetic samples
cvae.eval()
with torch.no_grad():
    #Random latent + random one-hot
    z = torch.randn(1000, latent_dim)
    labels = torch.eye(num_classes)[torch.randint(0, num_classes, (1000,))]
    synthetic_samples = cvae.decode(z, labels).cpu().numpy()
    synthetic_labels = np.argmax(labels.numpy(), axis=1)

unique_orig, counts_orig = np.unique(y_train, return_counts=True)
orig_counts_dict = dict(zip(unique_orig, counts_orig))

from collections import Counter
original_counts = Counter(y_train)
print("Original class counts:", original_counts)
smote = SMOTE(sampling_strategy='auto', random_state=42)
encoded_train_data, _ = cvae.encode(
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(y_train_onehot, dtype=torch.float32)
)
encoded_train_data = encoded_train_data.detach().numpy()
smote_latent, smote_labels = smote.fit_resample(encoded_train_data, y_train)
decoded_smote = cvae.decode(
    torch.tensor(smote_latent, dtype=torch.float32),
    torch.tensor(np.eye(num_classes)[smote_labels], dtype=torch.float32)
).detach().numpy()
target_count = 30
additional_samples_needed = {cls: target_count - original_counts[cls] for cls in original_counts if original_counts[cls] < target_count}
print("Additional samples needed:", additional_samples_needed)

selected_smote_samples = []
selected_smote_labels = []
for cls, count in additional_samples_needed.items():
    smote_indices = np.where(smote_labels == cls)[0]
    selected_indices = np.random.choice(smote_indices, count, replace=False)
    selected_smote_samples.append(decoded_smote[selected_indices])
    selected_smote_labels.append(smote_labels[selected_indices])

X_train_combined = np.concatenate([X_train] + selected_smote_samples, axis=0)
y_train_combined = np.concatenate([y_train] + selected_smote_labels, axis=0)

save_dir = r"/DS_andoriginal18x18NONSA"
os.makedirs(save_dir, exist_ok=True)

#Function to save samples with the specified naming convention
def save_dataset_with_custom_names(X, y, is_ds_sample, save_dir, label_encoder, dates):
    for i, (sample, label, ds_flag) in enumerate(zip(X, y, is_ds_sample)):
        #Decode the class label to its string name
        class_name = f"[{label_encoder.inverse_transform([label])[0]}]"

        #Determine the date
        date_collected = "NA" if ds_flag else dates[i]

        #Determine the sample number
        sample_number = f"DS{i}" if ds_flag else f"{i}"

        #Create the filename and save
        filename = f"{class_name}_{date_collected}_gaf_image_{sample_number}.npy"
        file_path = os.path.join(save_dir, filename)
        np.save(file_path, sample)

#Generate a boolean array indicating whether each sample is a DS sample
is_ds_sample = np.array([False] * len(X_train) + [True] * len(decoded_smote))

#Generate a list of dates for the original samples (use 'NA' for DS samples) and save
dates_original = [f"2022-08-11T05-24-40.{i:02d}" for i in range(len(X_train))]
dates_ds = ["NA"] * len(decoded_smote)
dates_combined = dates_original + dates_ds
save_dataset_with_custom_names(
    X_train_combined, y_train_combined, is_ds_sample, save_dir, label_encoder, dates_combined
)

print(f"Dataset saved in {save_dir}")

unique_orig, counts_orig = np.unique(y_train, return_counts=True)
print("Original training set class counts:")
for c, num in zip(unique_orig, counts_orig):
    print(f"Class {c}, Count: {num}")

unique_comb, counts_comb = np.unique(y_train_combined, return_counts=True)
print("Combined training set class counts:")
for c, num in zip(unique_comb, counts_comb):
    print(f"Class {c}, Count: {num}")

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

original_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
original_classifier.fit(X_train_flat, y_train)
y_pred_original = original_classifier.predict(X_test_flat)
conf_matrix0 = confusion_matrix(y_test, y_pred_original)
acsa0 = compute_acsa(conf_matrix0)
gm0 = compute_gm(conf_matrix0)
fm0 = compute_fm(conf_matrix0)

print("Confusion Matrix:")
print(conf_matrix0)
print(f"Average Class Specific Accuracy (ACSA): {acsa0:.4f}")
print(f"Macro-Averaged Geometric Mean (GM): {gm0:.4f}")
print(f"Macro-Averaged F1 Measure (FM): {fm0:.4f}")
print("Original dataset report:")
print(classification_report(y_test, y_pred_original))

#Combined
X_train_combined_flat = X_train_combined.reshape(X_train_combined.shape[0], -1)
combined_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
combined_classifier.fit(X_train_combined_flat, y_train_combined)
y_pred_combined = combined_classifier.predict(X_test_flat)
conf_matrix = confusion_matrix(y_test, y_pred_combined)
acsa = compute_acsa(conf_matrix)
gm = compute_gm(conf_matrix)
fm = compute_fm(conf_matrix)

print("Confusion Matrix:")
print(conf_matrix)
print(f"Average Class Specific Accuracy (ACSA): {acsa:.4f}")
print(f"Macro-Averaged Geometric Mean (GM): {gm:.4f}")
print(f"Macro-Averaged F1 Measure (FM): {fm:.4f}")
print("Combined dataset report:")
print(classification_report(y_test, y_pred_combined))

selected_class = 0
original_sample_index = np.where(y_train == selected_class)[0][0]
original_sample = X_train[original_sample_index]
smote_sample_index = np.where(smote_labels == selected_class)[0][0]
smote_sample = decoded_smote[smote_sample_index]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_sample[0], cmap='gray')
axes[0].set_title('Original Sample')
axes[0].axis('off')
axes[1].imshow(smote_sample[0], cmap='gray')
axes[1].set_title('SMOTE Sample')
axes[1].axis('off')
plt.show()

def plot_class_distribution_tsne(X, y, title):
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X)
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab20', s=50)
    plt.colorbar(scatter, ticks=range(num_classes), label='Class')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.title(title)
    plt.show()

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_train_combined_flat = X_train_combined.reshape(X_train_combined.shape[0], -1)

plot_class_distribution_tsne(X_train_flat, y_train, 'Original Class Distribution')
plot_class_distribution_tsne(X_train_combined_flat, y_train_combined, 'Combined Class Distribution')

def display_original_and_ds_samples(X_train, y_train, decoded_smote, smote_labels, num_classes):
    fig, axes = plt.subplots(num_classes, 2, figsize=(10, num_classes * 3))
    for class_label in range(num_classes):
        #random original sample from the class
        original_indices = np.where(y_train == class_label)[0]
        random_original_index = np.random.choice(original_indices)
        original_sample = X_train[random_original_index]

        #random DS sample from the same class
        smote_indices = np.where(smote_labels == class_label)[0]
        random_smote_index = np.random.choice(smote_indices)
        smote_sample = decoded_smote[random_smote_index]

        axes[class_label, 0].imshow(original_sample[0], cmap='gray')
        axes[class_label, 0].set_title(f'Original Class {class_label}')
        axes[class_label, 0].axis('off')

        axes[class_label, 1].imshow(smote_sample[0], cmap='gray')
        axes[class_label, 1].set_title(f'DS Class {class_label}')
        axes[class_label, 1].axis('off')

    plt.tight_layout()
    plt.show()

display_original_and_ds_samples(X_train, y_train, decoded_smote, smote_labels, num_classes)